In [ ]:
from astropy import units as u
from astropy.coordinates import SkyCoord

import matplotlib.pyplot as plt
from IPython.display import display
from gammapy.data import EventList
from gammapy.datasets import Datasets, MapDataset
from gammapy.irf import EDispKernelMap, PSFMap
from gammapy.maps import Map, MapAxis, WcsGeom
from gammapy.modeling import Fit
from matplotlib.patches import Circle
from astropy.table import vstack
from gammapy.data import EventList
from gammapy.catalog import SourceCatalog4FGL

import yaml
from fermipy.gtanalysis import GTAnalysis
from fermipy.plotting import ROIPlotter, SEDPlotter

catalog_4fgl = SourceCatalog4FGL("/Users/vdk/shared/fermibottle/gll_psc_v35.fit")

# Info about our initial catalogue source

In [ ]:
catalog_4fgl['4FGL J1908.6+0915e']

# Set up of the analysis

In [ ]:
gta = GTAnalysis('/Users/vdk/software/my_codebase/python/data_analysis/fermi_lat/sgr1900/fermi_config_sgr1900_15deg_PSF1-3_two_sources_GAL.yaml', logging={'verbosity': 3})

In [ ]:
gta.setup()

In [ ]:
gta.print_model()

In [ ]:
plt.figure(figsize=(20, 20))
gta.write_roi('initial',make_plots=True,save_model_map=True)
plt.show()

## Free parameters of the sources in the region 3 degrees around the source

In [ ]:
gta.free_sources(free=False)
gta.free_sources(skydir=gta.roi[gta.roi.sources[0].name].skydir,distance=[3.0],free=True)

In [ ]:
gta.optimize()
gta.fit()

In [ ]:
gta.print_model()

In [ ]:
plt.clf()
tsmap_postfit = gta.tsmap(prefix='TSmap_start',make_plots=True,write_fits=True,write_npy=True)

In [ ]:
p = ROIPlotter(tsmap_postfit['sqrt_ts'], roi=gta.roi, 
               cmap='ds9_b',   # example colormap
               graticule_radii=[0.6],  # optional, for overlaying circles
               label_ts_threshold=5.0)  # optional threshold for labeling

snr_g_043 = SkyCoord(43.011, 0.746, unit="deg", frame="galactic")

plt.figure(figsize=(10, 10))  # e.g., 10x10 inches
p.plot(interpolation='bicubic', zoom=6,vmin=3,vmax=5, levels=[0,3,5,7])
p.draw_circle(0.11,  # radius in degrees
              skydir=snr_g_043,
              edgecolor='cyan',
              linewidth=2.0,
              linestyle='--')

plt.title("$\sqrt{TS}$ map of the magnetar region")
plt.show()

# Exluding catalogue source from the model to see distribution of TS

In [ ]:
plt.clf()
plt.clf()
tsmap_postfit_no_source = gta.tsmap(
    prefix='TSmap_no_source',
    make_plots=True,
    write_fits=True,
    write_npy=True, 
    exclude=gta.roi.sources[0].name)

In [ ]:
p = ROIPlotter(tsmap_postfit_no_source['sqrt_ts'], roi=gta.roi, 
               cmap='ds9_b',   # example colormap
               graticule_radii=[0.6],  # optional, for overlaying circles
               label_ts_threshold=5.0)  # optional threshold for labeling

snr_g_043 = SkyCoord(43.011, 0.746, unit="deg", frame="galactic")

plt.figure(figsize=(10, 10))  # e.g., 10x10 inches
p.plot(interpolation='bicubic', zoom=7,vmin=3,vmax=5.5, levels=[0,3,5,7])
p.draw_circle(0.11,  # radius in degrees
              skydir=snr_g_043,
              edgecolor='cyan',
              linewidth=2.0,
              linestyle='--')

plt.title("$\sqrt{TS}$ map of the magnetar region, no source in the model")
plt.show()

# Split of the central source into two components

## Two extended sources

In [ ]:
glon1, glat1 = 43.29, 0.2  # Bottom red circle
glon2, glat2 = 43.00,  0.47  # Central blob

c1 = SkyCoord(l=glon1*u.deg, b=glat1*u.deg, frame='galactic')
c2 = SkyCoord(l=glon2*u.deg, b=glat2*u.deg, frame='galactic')



### First need to remove the source completely from the model

In [ ]:
gta.delete_source('4FGL J1908.6+0915e') 

### Now lets add two new sources and fit them to check how big is the likelihood of detection ($\sqrt{TS}$) of such sources in the region

In [ ]:
gta.add_source(f"Src1_extended", {
        'type':         'RadialDisc',
        'radius':        0.18,
        'glon':          c1.l.to_value(), 
        'glat':          c1.b.to_value(),
        'spectrum_type':'PowerLaw',
        'Index':        2.2,
        'Prefactor':    1e-13,
        'Scale':        1000.0
    })
gta.add_source(f"Src2_extended", {
        'type':         'RadialDisc',
        'radius':        0.18,
        'glon':          c2.l.to_value(), 
        'glat':          c2.b.to_value(),
        'spectrum_type':'PowerLaw',
        'Index':        2.2,
        'Prefactor':    1e-13,
        'Scale':        1000.0
    })

gta.print_model()

### Look on the sources that we added

In [ ]:
p = ROIPlotter(tsmap_postfit_no_source['sqrt_ts'], roi=gta.roi, 
               cmap='ds9_b',   # example colormap
               graticule_radii=[0.6],  # optional, for overlaying circles
               label_ts_threshold=5.0)  # optional threshold for labeling

snr_g_043 = SkyCoord(43.011, 0.746, unit="deg", frame="galactic")

plt.figure(figsize=(10, 10))  # e.g., 10x10 inches
p.plot(interpolation='bicubic', zoom=7,vmin=3,vmax=5.5, levels=[0,3,5,7])
p.draw_circle(0.11,  # radius in degrees
              skydir=snr_g_043,
              edgecolor='cyan',
              linewidth=2.0,
              linestyle='--')

p.draw_circle(0.18,  # radius in degrees
              skydir=c1,
              edgecolor='green',
              linewidth=2.0,
              linestyle='--')

p.draw_circle(0.18,  # radius in degrees
              skydir=c2,
              edgecolor='green',
              linewidth=2.0,
              linestyle='--')

plt.title("$\sqrt{TS}$ map of the magnetar region, with extended sources")
plt.show()

### Fit new model

In [ ]:
gta.optimize()
gta.fit()

In [ ]:
gta.print_model()

In [ ]:
plt.clf()
plt.clf()
tsmap_postfit_with_2_extended_sources = gta.tsmap(
    prefix='TSmap_2extended_sources',
    make_plots=True,
    write_fits=True,
    write_npy=True,)

In [ ]:
p = ROIPlotter(tsmap_postfit_with_2_extended_sources['sqrt_ts'], roi=gta.roi, 
               cmap='ds9_b',   # example colormap
               graticule_radii=[0.6],  # optional, for overlaying circles
               label_ts_threshold=5.0)  # optional threshold for labeling

snr_g_043 = SkyCoord(43.011, 0.746, unit="deg", frame="galactic")

plt.figure(figsize=(10, 10))  # e.g., 10x10 inches
p.plot(interpolation="bilinear", zoom=8,vmin=3,vmax=5.5, levels=[0,3,4,5])
p.draw_circle(0.11,  # radius in degrees
              skydir=snr_g_043,
              edgecolor='cyan',
              linewidth=2.0,
              linestyle='--')


glon1, glat1 = 43.29, 0.2  # Bottom red circle
glon2, glat2 = 43.00,  0.47  # Central blob

c1 = SkyCoord(l=glon1*u.deg, b=glat1*u.deg, frame='galactic')
c2 = SkyCoord(l=glon2*u.deg, b=glat2*u.deg, frame='galactic')

p.draw_circle(0.18,  # radius in degrees
              skydir=c1,
              edgecolor='green',
              linewidth=2.0,
              linestyle='--')

p.draw_circle(0.18,  # radius in degrees
              skydir=c2,
              edgecolor='green',
              linewidth=2.0,
              linestyle='--')

plt.title("$\sqrt{TS}$ residual map for the magnetar region and two extended sources")
plt.show()

# Now we can find the spectrum of the source